In [1]:
import pandas as pd
import numpy as np
data = {
    "age": [25, 40, None, 50, 23, None, 48, 33],
    "fever": [1, 0, 1, None, 0, 1, 0, 1],
    "cough": [1, 0, 1, 0, None, 1, 1, 1],
    "sick": [1, 0, 1, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)
print("Original DataFrame:")
print(df)

Original DataFrame:
    age  fever  cough  sick
0  25.0    1.0    1.0     1
1  40.0    0.0    0.0     0
2   NaN    1.0    1.0     1
3  50.0    NaN    0.0     1
4  23.0    0.0    NaN     0
5   NaN    1.0    1.0     1
6  48.0    0.0    1.0     0
7  33.0    1.0    1.0     1


In [2]:
from sklearn.impute import SimpleImputer

# Impute missing values only for preview: mean for 'age', most frequent for binary columns
age_imputer = SimpleImputer(strategy='mean')
binary_imputer = SimpleImputer(strategy='most_frequent')

df_imputed_preview = df.copy()
df_imputed_preview[['age']] = age_imputer.fit_transform(df_imputed_preview[['age']])
df_imputed_preview[['fever', 'cough']] = binary_imputer.fit_transform(df_imputed_preview[['fever', 'cough']])

print("DataFrame after imputation preview:")
print(df_imputed_preview)

DataFrame after imputation preview:
    age  fever  cough  sick
0  25.0    1.0    1.0     1
1  40.0    0.0    0.0     0
2  36.5    1.0    1.0     1
3  50.0    1.0    0.0     1
4  23.0    0.0    1.0     0
5  36.5    1.0    1.0     1
6  48.0    0.0    1.0     0
7  33.0    1.0    1.0     1


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Build a preprocessor with per-column imputation strategies
preprocessor = ColumnTransformer(
    transformers=[
        ('age_imputer', SimpleImputer(strategy='mean'), ['age']),
        ('binary_imputer', SimpleImputer(strategy='most_frequent'), ['fever', 'cough'])
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

print("Pipeline created successfully:")
print(pipeline)

Pipeline created successfully:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('age_imputer',
                                                  SimpleImputer(), ['age']),
                                                 ('binary_imputer',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  ['fever', 'cough'])])),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])


In [4]:
X = df[['age', 'fever', 'cough']]
y = df['sick']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Training data:")
print(X_train)
print("Training labels:")
print(y_train)
print("Testing data:")
print(X_test)
print("Testing labels:")
print(y_test)

Training data:
    age  fever  cough
0  25.0    1.0    1.0
7  33.0    1.0    1.0
2   NaN    1.0    1.0
4  23.0    0.0    NaN
3  50.0    NaN    0.0
6  48.0    0.0    1.0
Training labels:
0    1
7    1
2    1
4    0
3    1
6    0
Name: sick, dtype: int64
Testing data:
    age  fever  cough
1  40.0    0.0    0.0
5   NaN    1.0    1.0
Testing labels:
1    0
5    1
Name: sick, dtype: int64


In [5]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print("Predictions on the test set:")   
print(y_pred)

Predictions on the test set:
[1 1]


In [7]:
from sklearn.metrics import classification_report
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.50      1.00      0.67         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2

